In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import sklearn as sk
import requests

# Configurações de display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Configurar endpoint da API do BACEN
import sgs
sgs.api = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados'

## Anbima Fundamentos de Economia e Finanças

### Fundamentos de Economia

 ##### Principais indicadores econômicos do Brasil
 São ferramentas usadas para compreender e avaliar o status econômico atual, assim como permite também comparar com períodos históricos. Os indicadores podem ser desenvolvidos por participantes do mercado sistema financeiro, instituições públicas e privadas (CVM, BACEN, FGV, etc.). Os principais indicadores são: taxa de juros, inflação, produtos e câmbio.
 

##### Indicadores de taxa de Juros

1.1 SELIC (meta):
- Definido em reunião do COPOM (Comitê de Política Monetária) a cada 45 dias
- Tem como objetivo definir a taxa de juros que melhor contenha a inflação para equilibrar o consumo, o crédito e a atividade econômica, visando a estabilidade de preços
- Trabalha para cumprir as metas definidas pelo CMN (Conselho Monetário Nacional) de inflação
- O Comitê é formado pelo Presidente e diretores do Banco Central do Brasil

In [7]:
SELIC_atual = 0.13
SELIC_anterior = 0.14

def calcular_variacao_SELIC():
    valor_variacao = (SELIC_atual - SELIC_anterior) / SELIC_anterior
    if SELIC_atual > SELIC_anterior:
        print(f"A taxa SELIC aumentou em {valor_variacao:.2%}, portanto sua política é contracionista. \n"
              f"Significa que os juros estão mais altos, o que pode levar a um aumento nos custos de empréstimos e financiamentos.\n"
              f"Adicionalmente, incentiva investimentos de menor risco.")

    elif SELIC_atual < SELIC_anterior:
        print(f"A taxa SELIC diminuiu em {valor_variacao:.2%}, portanto sua política é expansionista. \n"
              f"Significa que os juros estão mais baixos, o que pode incentivar o consumo e os investimentos de maior risco.")

    else:
        print("A taxa SELIC permaneceu inalterada, indicando uma política monetária neutra. \n"
              "Isso pode sugerir estabilidade econômica ou receio/incertezas do cenário político/econômico externo.")
              
indicador_SELIC = calcular_variacao_SELIC()
indicador_SELIC

A taxa SELIC diminuiu em -7.14%, portanto sua política é expansionista. 
Significa que os juros estão mais baixos, o que pode incentivar o consumo e os investimentos de maior risco.


1.2 SELIC Over (efetiva):
- Divulgada diariamente pelo BACEN
- É a taxa média dos juros das operações de empréstimo de curtíssimo prazo (um dia para o outro, ou seja, overnight) entre bancos, lastreadas em títulos públicos federais no Sistema Selic
- Não é definida diretamente, ela é apurada diariamente com base nas operações reais entre os bancos, ficando próxima a Selic meta
- Em termos simples, é o custo do dinheiro entre bancos de um dia para o outro

1.3 Selic DI (ou taxa DI / CDI):
- Divulgada diariamente pela B3
- É a taxa média dos empréstimos overnight entre bancos sem garantia em títulos públicos (CDI)

1.4 Funcionamento na prática:

De modo geral, as taxas meta, over e DI tem valores muitos próximos, isso acontece porque o BACEN atua para isso.
- Meta: taxa que o BACEN quer
- Over: taxa que o mercado faz
- DI/CDI: taxa que o investidor opera

In [6]:
data_inicio = '01/01/2020'
data_fim = '31/12/2020'

def get_selic_meta(data_inicio: str, data_fim: str) -> pd.DataFrame:
    """
    Retorna a série histórica da Selic Meta entre duas datas.
    """
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial={data_inicio}&dataFinal={data_fim}"
    
    response = requests.get(url)
    response.raise_for_status()
    
    df = pd.DataFrame(response.json())
    df['data'] = pd.to_datetime(df['data'], dayfirst=True)
    df['valor'] = df['valor'].astype(float)
    
    return df


def get_selic_over(data_inicio: str, data_fim: str) -> pd.DataFrame:
    """
    Retorna a série histórica da Selic Over entre duas datas.
    """
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.1178/dados?formato=json&dataInicial={data_inicio}&dataFinal={data_fim}"
    
    response = requests.get(url)
    response.raise_for_status()
    
    df = pd.DataFrame(response.json())
    df['data'] = pd.to_datetime(df['data'], dayfirst=True)
    df['valor'] = df['valor'].astype(float)
    
    return df

df_selic_meta = get_selic_meta(data_inicio, data_fim)
print(df_selic_meta.head())
df_selic_over = get_selic_over(data_inicio, data_fim)
print(df_selic_over.head())

        data  valor
0 2020-01-01   4.50
1 2020-01-02   4.50
2 2020-01-03   4.50
3 2020-01-04   4.50
4 2020-01-05   4.50
        data  valor
0 2020-01-02   4.40
1 2020-01-03   4.40
2 2020-01-06   4.40
3 2020-01-07   4.40
4 2020-01-08   4.40


1.5 Operações compromissadas:

São as operações de títulos públicos feitos entre o BACEN e o mercado financeiros onde há promessa de operação inversa no dia seguinte. O objetivo é, de acordo com a liquidez do mercado (bancos com dinheiro sobrando ou faltando), comprar/vender títulos e fazer a operação inversa no dia seguinte, visando trazer a taxa SELIC over para algo próximo à SELIC meta. O Bacen não espera sair o “número oficial”, ele já vê o comportamento do mercado em tempo real e ajusta a liquidez antes da taxa descolar da meta.

Exemplo 1: SELIC meta está em 15%. Um título SELIC é negociado a 14,80%, logo, o BACEN identifica que o valor está abaixo da SELIC meta, o que significa que há maior liquidez no mercado (lógica: estão negociando a um % menor do que poderiam). Dessa forma, o BACEN faz uma venda compromissada, estimulando a % a se aproximar de 15% (bancos estão com dinheiro sobrando no mercado, BACEN retira).

Exemplo 2: SELIC meta está em 15%. Um título SELIC é negociado a 15,20%, logo, o BACEN identifica que o valor está acima da SELIC meta, o que significa que há menor liquidez no mercado (lógica: estão negociando a um % maior do que poderiam). Dessa forma, o BACEN faz uma compra compromissada, estimulando a % a se aproximar de 15% (bancos estão com dinheiro faltando no mercado, BACEN injeta).


Exemplo de funcionamento ao longo do dia:

Bancos iniciam o dia avaliando reservas, pagamentos e posição de caixa, o que já indica se haverá excesso ou falta de liquidez no sistema; o Banco Central do Brasil acompanha isso de forma mais completa. Ao longo da manhã, começam as operações overnight entre bancos e a taxa praticada passa a sinalizar desvios em relação à meta (abaixo indica excesso de liquidez, acima indica escassez). Se necessário, o Bacen atua por meio de leilões de operações compromissadas, definindo tipo (compra ou venda), volume e prazo, com impacto imediato nas reservas: venda retira liquidez, compra injeta. Durante a tarde, novas liquidações e ajustes de caixa ocorrem, podendo demandar atuações adicionais. No fim do dia, os bancos fecham suas posições e as últimas operações determinam as taxas efetivas; depois disso, o Bacen calcula a média ponderada dessas operações, gerando a Selic Over, que é divulgada no dia seguinte. Exemplo: com meta de 10,50%, se pela manhã a taxa estiver em 10,30%, o Bacen realiza venda compromissada para reduzir liquidez, levando a taxa de volta para perto da meta até o fechamento.

In [8]:
def simular_selic_over(
    dias=30,
    selic_meta=0.105,          # 10,5% ao ano
    liquidez_inicial=1000,     # nível de reservas do sistema
    sensibilidade=0.00005,     # impacto da liquidez na taxa
    intervencao_intensidade=200  # quanto o Bacen atua
):
    """
    Simula a dinâmica da Selic Over com intervenções via compromissadas.
    """
    
    liquidez = liquidez_inicial
    resultados = []

    for dia in range(dias):
        
        # choque aleatório de liquidez (pagamentos, fluxos etc.)
        choque = np.random.normal(0, 50)
        liquidez += choque
        
        # taxa reage à liquidez (mais liquidez -> taxa menor)
        selic_over = selic_meta - sensibilidade * (liquidez - liquidez_inicial)
        
        intervencao = 0
        
        # regra do Bacen
        if selic_over < selic_meta:
            # excesso de liquidez -> venda compromissada
            intervencao = -intervencao_intensidade
            liquidez += intervencao
        
        elif selic_over > selic_meta:
            # falta de liquidez -> compra compromissada
            intervencao = intervencao_intensidade
            liquidez += intervencao
        
        # nova taxa após intervenção
        selic_over_ajustada = selic_meta - sensibilidade * (liquidez - liquidez_inicial)
        
        resultados.append({
            "dia": dia,
            "liquidez": liquidez,
            "df_selic_meta": selic_meta,
            "selic_over_antes": selic_over,
            "intervencao_bacen": intervencao,
            "selic_over_depois": selic_over_ajustada
        })
    
    return pd.DataFrame(resultados)

df = simular_selic_over(dias=10)

print(df.head())

   dia  liquidez  df_selic_meta  selic_over_antes  intervencao_bacen  \
0    0   1193.19           0.10              0.11                200   
1    1   1047.36           0.10              0.09               -200   
2    2    874.96           0.10              0.10               -200   
3    3   1195.53           0.10              0.11                200   
4    4   1041.95           0.10              0.09               -200   

   selic_over_depois  
0               0.10  
1               0.10  
2               0.11  
3               0.10  
4               0.10  


##### Indicadores de taxa de Inflação
IPCA:


IPCA-15


IGPM:

##### Indicadores de taxa de Produto
PIB: 




##### Indicadores de taxa de Câmbio